# ViHSD v2 — verified recovery of completed training

The GPU training ran for **five epochs** in `ndhoang2310/vihsd-real-noise-augmentation-v2/1`. It saved all primary, baseline and FGM epsilon1 evaluations, then failed when loading an unavailable **optional pilot augmentation reference**. The original platform status remains ERROR. This CPU notebook independently verifies and exports the completed outputs; it does not retrain or conceal the original error. Weight checksum, original source/log, five-epoch history, predictions and reconstruction provenance are included. Synthetic toxic labels remain unverified by humans.


The full research, original EDA and four-checkpoint comparison are included under `reports/delivery/`. The execution plan is a historical draft; `ROBUSTNESS_COMPLETE.json` and `KAGGLE_RECOVERY_VERIFIED.json` record actual verification. The report explicitly retains one-training-seed and synthetic-label limitations.

In [ ]:
from pathlib import Path
import json,zipfile,hashlib
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score,accuracy_score,confusion_matrix
import shutil
archives=list(Path('/kaggle/input').rglob('recovery.zip'))
root=Path('/kaggle/working/robustness_project')
if archives:
    with zipfile.ZipFile(archives[0]) as z:z.extractall('/kaggle/working')
else:
    source=next(Path('/kaggle/input').rglob('ROBUSTNESS_COMPLETE.json')).parent
    shutil.copytree(source,root,dirs_exist_ok=True)
done=json.loads((root/'ROBUSTNESS_COMPLETE.json').read_text())
assert done['status']=='complete_recovered'
weights=root/'runs'/done['run_name']/'best/model.safetensors'
assert hashlib.file_digest(weights.open('rb'),'sha256').hexdigest()==done['best_model_sha256']
gold=pd.read_parquet(root/'official_test_labels.parquet').set_index('sample_id')
rows=[]
for model in [done['condition'],'baseline']+done['extra_references']:
 summary=json.loads((root/'robustness'/model/'summary.json').read_text())
 assert len(summary)==11
 for profile,m in summary.items():
  p=pd.read_parquet(root/'robustness'/model/f'{profile}_predictions.parquet')
  assert len(p)==6680 and p.sample_id.is_unique
  assert np.array_equal(p.label,gold.loc[p.sample_id].label)
  prob=p[['p_clean','p_offensive','p_hate']].to_numpy()
  assert np.isfinite(prob).all() and np.allclose(prob.sum(1),1,atol=1e-5)
  assert np.array_equal(prob.argmax(1),p.prediction)
  f1=f1_score(p.label,p.prediction,labels=[0,1,2],average='macro',zero_division=0)
  assert abs(f1-m['macro_f1'])<1e-12
  assert abs(accuracy_score(p.label,p.prediction)-m['accuracy'])<1e-12
  assert confusion_matrix(p.label,p.prediction,labels=[0,1,2]).tolist()==m['confusion_matrix']
  rows.append({'model':model,'profile':profile,'macro_f1':f1,'accuracy':m['accuracy']})
result={'passed':True,'profiles_recomputed':len(rows),'training_occurred_in':done['training_kernel'],'training_platform_status':done['training_kernel_status'],'recovery':'CPU verification and archival export only; no retraining','model_sha256':done['best_model_sha256']}
(root/'KAGGLE_RECOVERY_VERIFIED.json').write_text(json.dumps(result,indent=2))
display(pd.DataFrame(rows))
print(json.dumps(result,indent=2))


In [ ]:
from IPython.display import HTML, display
display(HTML((root/'reports/delivery/ROBUSTNESS.html').read_text()))